$$
\max Z = \sum_{j : (s,j) \in E} x_{s,j}
$$

$$
0 \le x_{i,j} \le C_{i,j}, \qquad \forall (i,j) \in E
$$

$$
\sum_{i : (i,k) \in E} x_{i,k} - \sum_{j : (k,j) \in E} x_{k,j} = 0, \qquad \forall k \in V \setminus \lbrace s,t \rbrace
$$

$$
x_{i,j} \ge 0, \qquad \forall (i,j) \in E
$$


**Sets:** $V$ (nodes), $E$ (edges), $s$ (source), $t$ (sink)  
**Parameters:** $C_{i,j}$ (capacity)  
**Variables:** $x_{i,j}$ (flow)

In [228]:
from pyomo.environ import *
from ortools.sat.python import cp_model
import warnings
warnings.filterwarnings('ignore')

In [246]:
capacity_data = {
    (0, 1): 10,
    (0, 2): 15,
    (1, 2): 5,
    (1, 3): 8,
    (2, 4): 10,
    (3, 4): 12,
}
nodes = [0, 1, 2, 3, 4]
edges = list(capacity_data.keys())
source = 0
sink = 4


In [247]:
def lp():
    model = ConcreteModel()
    model.V = Set(initialize=nodes)
    model.E = Set(initialize=edges)
    
    def flow_bounds(model, i, j):
        return (0, capacity_data[i,j])

    def obj_rule(model):
        return sum(model.x[s,j] for (s,j) in model.E if s == source)
        
    def flow_balance_rule(model, k):
        if k == source or k == sink:
            return Constraint.Skip
    
        inflow = 0
        for i,j in model.E:
            if j == k:
                inflow += model.x[i,j]
        # inflow = quicksum(model.x[i,j] for i,j in model.E if j == k)
    
        outflow = 0
        for i,j in model.E:
            if i == k:
                outflow += model.x[i,j]
    
        return inflow == outflow
        
    def capacity_rule(model, i, j):
        """ We set upper bounds on x[i, j] directly in the variable definition,
        so this capacity constraint is not needed. """
        return (model.x[i,j] <= capacity_data[i,j])

        
    
    model.x = Var(model.E, domain=NonNegativeReals, bounds=flow_bounds)
    model.obj = Objective(rule=obj_rule, sense=maximize)
    model.balance = Constraint(model.V, rule=flow_balance_rule)
    # model.capacity = Constraint(model.E, rule=capacity_rule)

    solver = SolverFactory("glpk")
    results = solver.solve(model, tee=False)
    print(results.solver.status)
    print(value(model.obj))
    # print(model.x.pprint())

In [251]:
%time lp()

ok
18.0
CPU times: user 8.86 ms, sys: 3 ms, total: 11.9 ms
Wall time: 13.1 ms


In [252]:
def cp():

    model = cp_model.CpModel()
    x = {}
    for u,v in edges:
        cap = capacity_data[u,v]
        x[u, v] = model.NewIntVar(0, capacity_data[u,v], f"x{u}{v}")

    for k in nodes:
        if k!=source and k!=sink:
            inflow = [] 
            outflow = [] 
            for u,v in edges:
                if v == k:
                    inflow.append(x[u,v])
                if u == k:
                    outflow.append(x[u,v])
    
            model.Add(sum(inflow) == sum(outflow))
            
    out_from_source = [x[u,v] for u,v in edges if u == source]
    model.Maximize(sum(out_from_source))

    
    solver = cp_model.CpSolver()
    results = solver.Solve(model)
   
    print(results.FEASIBLE)
    print(solver.ObjectiveValue())
    # for u, v in edges:
    #     val = solver.Value(x[u, v])
    #     cap = capacity_data[u, v]
    #     print(f"{u}=>{v}: {val}")

In [253]:
%time cp()

CpSolverStatus.FEASIBLE
18.0
CPU times: user 6.84 ms, sys: 1.99 ms, total: 8.83 ms
Wall time: 5.9 ms
